In [2]:
import sqlite3
import pandas as pd
import ast
import numpy as np
import os

In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F


In [4]:
db_path = os.path.abspath("cademycode.db")

In [5]:
spark = (
    SparkSession.builder
    .appName("SubscriberPipeline")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.jars.packages", "org.xerial:sqlite-jdbc:3.45.1.0")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/18 14:42:39 WARN Utils: Your hostname, MDs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.69.115.192 instead (on interface en0)
26/08/18 14:42:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/md_khalid/Data_Engineer/DataSet/codeacademy_dataset/subscriber-pipeline-starter-kit/dev/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/md_khalid/.ivy2.5.2/cache
The jars for the packages stored in: /Users/md_khalid/.ivy2.5.2/jars
org.xerial#sqlite-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c00e045e-b86b-4331-bff1-1b70d39eaa2b;1.0
	confs: [default]
	found org.xerial#sqlite-jdbc;3.45.1.0 in central
	found org.slf4j#slf4j-api;1.7.36 in central
downloading https://repo1.maven.org/mav

In [6]:
con = sqlite3.connect('cademycode.db')
cur = con.cursor()

In [7]:
table_list = [a for a in cur.execute("SELECT name from sqlite_MASTER WHERE type = 'table'")]
print(table_list)

[('cademycode_students',), ('cademycode_courses',), ('cademycode_student_jobs',)]


In [8]:
students = pd.read_sql_query("SELECT * FROM cademycode_students", con)
career_path = pd.read_sql_query("SELECT * FROM cademycode_courses", con)
student_jobs = pd.read_sql_query("SELECT * FROM cademycode_student_jobs",con)

In [9]:
print('students:', len(students))
print('career_path:', len(career_path))
print('student_jobs:', len(student_jobs))

students: 5000
career_path: 10
student_jobs: 13


In [10]:
students.head(100)

,uuid,name,dob,sex,contact_info,job_id,num_course_taken,current_career_path_id,time_spent_hrs
0,1,Annabelle Avery,1943-07-03,F,"{""mailing_address"": ""303 N Timber Key, Irondal...",7.0,6.0,1.0,4.99
1,2,Micah Rubio,1991-02-07,M,"{""mailing_address"": ""767 Crescent Fair, Shoals...",7.0,5.0,8.0,4.4
2,3,Hosea Dale,1989-12-07,M,"{""mailing_address"": ""P.O. Box 41269, St. Bonav...",7.0,8.0,8.0,6.74
3,4,Mariann Kirk,1988-07-31,F,"{""mailing_address"": ""517 SE Wintergreen Isle, ...",6.0,7.0,9.0,12.31
4,5,Lucio Alexander,1963-08-31,M,"{""mailing_address"": ""18 Cinder Cliff, Doyles b...",7.0,14.0,3.0,5.64
...,...,...,...,...,...,...,...,...,...
95,96,Myesha Dudley,1980-09-28,F,"{""mailing_address"": ""554 Merry Lagoon, Jewett,...",4.0,2.0,10.0,4.73
96,97,Moises Krekel,2004-06-15,M,"{""mailing_address"": ""455 Silent Smith, Radium ...",8.0,None,None,None
97,98,Martin Ramirez,1963-12-24,M,"{""mailing_address"": ""317 S Quay, Eldred villag...",6.0,0.0,6.0,4.07
98,99,Violette Mills,1999-11-04,F,"{""mailing_address"": ""165 Cliff Station, Radium...",1.0,5.0,4.0,19.25


In [11]:
df_students = students
df_career = career_path
df_jobs = student_jobs

In [28]:
spark.createDataFrame(df_students).schema

StructType([StructField('uuid', LongType(), True), StructField('name', StringType(), True), StructField('dob', StringType(), True), StructField('sex', StringType(), True), StructField('contact_info', StringType(), True), StructField('job_id', StringType(), True), StructField('num_course_taken', StringType(), True), StructField('current_career_path_id', StringType(), True), StructField('time_spent_hrs', StringType(), True)])

In [12]:
schema = types.StructType([
types.StructField('uuid', types.LongType(), True),
types.StructField('name', types.StringType(), True),
types.StructField('dob', types.StringType(), True), 
types.StructField('sex', types.StringType(), True), 
types.StructField('contact_info', types.StringType(), True),
types.StructField('job_id', types.StringType(), True), 
types.StructField('num_course_taken', types.StringType(), True), 
types.StructField('current_career_path_id', types.StringType(), True), 
types.StructField('time_spent_hrs', types.StringType(), True)])

In [18]:
spark_students = spark.createDataFrame(df_students.astype(str))
spark_courses = spark.createDataFrame(df_career.astype(str))
spark_jobs = spark.createDataFrame(df_jobs.astype(str))

In [19]:
spark_students = (
    spark.read.format("jdbc")
    .option("url", "jdbc:sqlite:cademycode.db")
    .option("dbtable", "cademycode_students")
    .option("driver", "org.sqlite.JDBC")
    .load()
)

spark_courses = (
    spark.read.format("jdbc")
    .option("url", f"jdbc:sqlite:{db_path}")
    .option("dbtable", "cademycode_courses")
    .option("driver", "org.sqlite.JDBC")
    .load()
)

spark_jobs = (
    spark.read.format("jdbc")
    .option("url", f"jdbc:sqlite:{db_path}")
    .option("dbtable", "cademycode_student_jobs")
    .option("driver", "org.sqlite.JDBC")
    .load()
)




In [20]:
spark_students.show(10)

26/08/18 14:48:56 ERROR Executor: Exception in task 0.0 in stage 1.0 (TID 1)
java.sql.SQLException: Bad value for type BigDecimal : {"mailing_address": "303 N Timber Key, Irondale, Wisconsin, 84736", "email": "annabelle_avery9376@woohoo.com"}
	at org.sqlite.jdbc3.JDBC3ResultSet.getBigDecimal(JDBC3ResultSet.java:169)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$makeGetter$3(JdbcUtils.scala:457)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$makeGetter$3$adapted(JdbcUtils.scala:455)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.getNextWithoutTiming(JdbcUtils.scala:382)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.$anonfun$getNext$1(JdbcUtils.scala:396)
	at org.apache.spark.sql.execution.metric.SQLMetrics$.withTimingNs(SQLMetrics.scala:234)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.getNext(JdbcUtils.scala:396)
	at org.apache.spark.sql.execution.datasources.jdbc.

Py4JJavaError: An error occurred while calling o92.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (10.69.115.192 executor driver): java.sql.SQLException: Bad value for type BigDecimal : {"mailing_address": "303 N Timber Key, Irondale, Wisconsin, 84736", "email": "annabelle_avery9376@woohoo.com"}
	at org.sqlite.jdbc3.JDBC3ResultSet.getBigDecimal(JDBC3ResultSet.java:169)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$makeGetter$3(JdbcUtils.scala:457)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$makeGetter$3$adapted(JdbcUtils.scala:455)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.getNextWithoutTiming(JdbcUtils.scala:382)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.$anonfun$getNext$1(JdbcUtils.scala:396)
	at org.apache.spark.sql.execution.metric.SQLMetrics$.withTi

In [18]:
student_jobs.head(20)

,job_id,job_category,avg_salary
0,1,analytics,86000
1,2,engineer,101000
2,3,software developer,110000
3,4,creative,66000
4,5,financial services,135000
5,6,education,61000
6,7,HR,80000
7,8,student,10000
8,9,healthcare,120000
9,0,other,80000


In [9]:

schema_ddl = """
    uuid BIGINT,
    name STRING,
    dob STRING,
    sex STRING,
    contact_info STRING,
    job_id INT,
    num_course_taken INT,
    current_career_path_id INT,
    time_spent_hrs DOUBLE
"""

# Convert Pandas DF (from SQLite) to Spark DF
spark_students = spark.createDataFrame(students, schema=schema_ddl)

/Users/md_khalid/Data_Engineer/DataSet/codeacademy_dataset/subscriber-pipeline-starter-kit/dev/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/md_khalid/Data_Engineer/DataSet/codeacademy_dataset/subscriber-pipeline-starter-kit/dev/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/md_khalid/Data_Engineer/DataSet/codeacademy_dataset/subscriber-pipeline-starter-kit/dev/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set t

PySparkTypeError: [FIELD_DATA_TYPE_UNACCEPTABLE_WITH_NAME] field job_id: IntegerType() can not accept object '7.0' in type <class 'str'>.

In [18]:
career_path.dtypes

career_path_id        int64
career_path_name     object
hours_to_complete     int64
dtype: object

In [19]:
student_jobs.dtypes

job_id           int64
job_category    object
avg_salary       int64
dtype: object